# Dataset 6: readmission, diagnosis, and utilization EDA

This split contains readmission, secondary diagnosis, prior outpatient utilization, admission source, and medication-status variables.

The notebook analyses this split independently. The original CSV is not changed; the cleaned copy is saved in `EDA_Dataset` for later joining on `encounter_id`.

## Setup

In [1]:
from pathlib import Path  # Import required library

import matplotlib.pyplot as plt  # Import required library
import pandas as pd  # Import required library
import seaborn as sns  # Import required library

sns.set_theme(style="whitegrid", context="notebook")  # Set the chart theme
pd.set_option("display.max_rows", 100)  # Configure dataframe display
pd.set_option("display.max_columns", 50)  # Configure dataframe display

In [2]:
def show_category_counts(series, title, top_n=15):  # Define the categorical-frequency chart helper
    counts = series.fillna("Missing").value_counts().head(top_n).sort_values()  # Calculate category frequencies
    fig, ax = plt.subplots(figsize=(10, max(3.5, 0.38 * len(counts))))  # Create the chart area
    ax.barh(counts.index.astype(str), counts.values, color="#4C78A8")  # Draw the horizontal bar chart
    ax.set_title(f"{title}: top {top_n} categories")  # Label the chart axis or title
    ax.set_xlabel("Encounter count")  # Label the chart axis or title
    ax.set_ylabel("")  # Label the chart axis or title
    for y, value in enumerate(counts.values):  # Add value labels to each bar
        ax.text(value, y, f" {value:,}", va="center", fontsize=8)  # Run this EDA or data-cleaning step
    plt.tight_layout()  # Adjust chart spacing
    plt.show()  # Display the chart


def show_numeric_distribution(series, title, discrete=False):  # Define the numeric-distribution chart helper
    values = series.dropna()  # Remove missing numeric values before plotting
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))  # Create the chart area
    sns.histplot(values, discrete=discrete, ax=axes[0], color="#4C78A8")  # Plot the feature distribution
    axes[0].set_title(f"{title}: distribution")  # Configure the numeric chart
    axes[0].set_xlabel(title)  # Configure the numeric chart
    sns.boxplot(x=values, ax=axes[1], color="#72B7B2")  # Plot the feature box plot
    axes[1].set_title(f"{title}: box plot")  # Configure the numeric chart
    axes[1].set_xlabel(title)  # Configure the numeric chart
    plt.tight_layout()  # Adjust chart spacing
    plt.show()  # Display the chart

## Load the source data

In [3]:
source_path = Path("divided_datasets") / "dataset_6.csv"  # Set the path to the source CSV
df_raw = pd.read_csv(source_path, na_values=["?"], keep_default_na=True, low_memory=False)  # Load the source CSV and convert question marks to missing values

## Initial inspection

In [4]:
df_raw.head()  # Inspect the original dataframe

,encounter_id,diag_2,readmitted,number_diagnoses,admission_source_id,glipizide-metformin,number_outpatient,acetohexamide,examide
0,2278392,NaN,NO,1,1,No,0,No,No
1,149190,250.01,>30,9,7,No,0,No,No
2,64410,250,NO,6,7,No,2,No,No
3,500364,250.43,NO,7,7,No,0,No,No
4,16680,157,NO,5,7,No,0,No,No


In [5]:
df_raw.info()  # Inspect the original dataframe

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype
---  ------               --------------   -----
 0   encounter_id         101766 non-null  int64
 1   diag_2               101408 non-null  str  
 2   readmitted           101766 non-null  str  
 3   number_diagnoses     101766 non-null  int64
 4   admission_source_id  101766 non-null  int64
 5   glipizide-metformin  101766 non-null  str  
 6   number_outpatient    101766 non-null  int64
 7   acetohexamide        101766 non-null  str  
 8   examide              101766 non-null  str  
dtypes: int64(4), str(5)
memory usage: 8.1 MB


In [6]:
df_raw.shape  # Inspect the original dataframe

(101766, 9)

In [7]:
df_raw.describe()  # Inspect the original dataframe

,encounter_id,number_diagnoses,admission_source_id,number_outpatient
count,1.017660e+05,101766.000000,101766.000000,101766.000000
mean,1.652016e+08,7.422607,5.754437,0.369357
std,1.026403e+08,1.933600,4.064081,1.267265
min,1.252200e+04,1.000000,1.000000,0.000000
25%,8.496119e+07,6.000000,1.000000,0.000000
50%,1.523890e+08,8.000000,7.000000,0.000000
75%,2.302709e+08,9.000000,7.000000,0.000000
max,4.438672e+08,16.000000,25.000000,42.000000


In [8]:
df_raw.isna().sum().sort_values(ascending=False).to_frame(name="missing_count")  # Inspect the original dataframe

,missing_count
diag_2,358
encounter_id,0
readmitted,0
number_diagnoses,0
admission_source_id,0
glipizide-metformin,0
number_outpatient,0
acetohexamide,0
examide,0


In [9]:
df_raw.duplicated().sum()  # Inspect the original dataframe

np.int64(0)

In [10]:
df_raw["encounter_id"].duplicated().sum()  # Run this EDA or data-cleaning step

np.int64(0)

In [11]:
df_raw["encounter_id"].is_unique  # Run this EDA or data-cleaning step

True

In [12]:
df_raw.nunique(dropna=False).sort_values().to_frame(name="unique_values_including_missing")  # Inspect the original dataframe

,unique_values_including_missing
examide,1
acetohexamide,2
glipizide-metformin,2
readmitted,3
number_diagnoses,16
admission_source_id,17
number_outpatient,39
diag_2,749
encounter_id,101766


## Cleaning decisions

- Missing-value normalization: Question-mark and blank text values are treated as missing.
- Secondary diagnosis: A missingness flag is retained and missing diagnosis codes are labeled Unknown for visualization.
- Constant feature removed: examide contains one observed value (No) and adds no analytical variation.
- Rows and join key: No rows are removed. encounter_id remains unchanged and unique for the later join.

In [13]:
df = df_raw.copy()  # Copy the source dataframe before cleaning

text_columns = df.select_dtypes(include=["object", "string"]).columns  # Identify text columns for normalization
for column in text_columns:  # Clean each text column
    df[column] = df[column].astype("string").str.strip().replace("", pd.NA)  # Strip whitespace and convert blank text to missing values

df["diag_2_missing"] = df["diag_2"].isna()  # Create or handle the diagnosis missingness information
df["diag_2"] = df["diag_2"].fillna("Unknown")  # Create or handle the diagnosis missingness information

assert df["examide"].nunique(dropna=False) == 1, "examide is expected to be constant"  # Validate the cleaning result
columns_removed = ["examide"]  # Record columns removed during cleaning
df = df.drop(columns=columns_removed)  # Remove the documented uninformative columns

assert len(df) == len(df_raw), "No encounter rows should be removed"  # Validate the cleaning result
assert df["encounter_id"].notna().all(), "The join key must not be missing"  # Validate the cleaning result
assert df["encounter_id"].is_unique, "The join key must remain unique"  # Validate the cleaning result
assert df.columns.is_unique, "Duplicate column names are not allowed"  # Validate the cleaning result

In [14]:
cleaning_summary = pd.DataFrame({  # Create the cleaning-decision summary
    "decision": ["Missing-value normalization","Secondary diagnosis","Constant feature removed","Rows and join key","Columns removed"],  # Add a cleaning-summary field
    "detail": ["Question-mark and blank text values are treated as missing.","A missingness flag is retained and missing diagnosis codes are labeled Unknown for visualization.","examide contains one observed value (No) and adds no analytical variation.","No rows are removed. encounter_id remains unchanged and unique for the later join.","Columns removed after review: examide"]  # Add a cleaning-summary field
})  # Run this EDA or data-cleaning step
cleaning_summary  # Create the cleaning-decision summary

,decision,detail
0,Missing-value normalization,Question-mark and blank text values are treate...
1,Secondary diagnosis,A missingness flag is retained and missing dia...
2,Constant feature removed,examide contains one observed value (No) and a...
3,Rows and join key,No rows are removed. encounter_id remains unch...
4,Columns removed,Columns removed after review: examide


## Post-cleaning validation

In [15]:
df.head()  # Run this EDA or data-cleaning step

,encounter_id,diag_2,readmitted,number_diagnoses,admission_source_id,glipizide-metformin,number_outpatient,acetohexamide,diag_2_missing
0,2278392,Unknown,NO,1,1,No,0,No,True
1,149190,250.01,>30,9,7,No,0,No,False
2,64410,250,NO,6,7,No,2,No,False
3,500364,250.43,NO,7,7,No,0,No,False
4,16680,157,NO,5,7,No,0,No,False


In [16]:
df.info()  # Run this EDA or data-cleaning step

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   encounter_id         101766 non-null  int64 
 1   diag_2               101766 non-null  string
 2   readmitted           101766 non-null  string
 3   number_diagnoses     101766 non-null  int64 
 4   admission_source_id  101766 non-null  int64 
 5   glipizide-metformin  101766 non-null  string
 6   number_outpatient    101766 non-null  int64 
 7   acetohexamide        101766 non-null  string
 8   diag_2_missing       101766 non-null  bool  
dtypes: bool(1), int64(4), string(4)
memory usage: 7.3 MB


In [17]:
df.shape  # Run this EDA or data-cleaning step

(101766, 9)

In [18]:
df.describe()  # Run this EDA or data-cleaning step

,encounter_id,number_diagnoses,admission_source_id,number_outpatient
count,1.017660e+05,101766.000000,101766.000000,101766.000000
mean,1.652016e+08,7.422607,5.754437,0.369357
std,1.026403e+08,1.933600,4.064081,1.267265
min,1.252200e+04,1.000000,1.000000,0.000000
25%,8.496119e+07,6.000000,1.000000,0.000000
50%,1.523890e+08,8.000000,7.000000,0.000000
75%,2.302709e+08,9.000000,7.000000,0.000000
max,4.438672e+08,16.000000,25.000000,42.000000


In [19]:
df.isna().sum().sort_values(ascending=False).to_frame(name="missing_count_after_cleaning")  # Run this EDA or data-cleaning step

,missing_count_after_cleaning
encounter_id,0
diag_2,0
readmitted,0
number_diagnoses,0
admission_source_id,0
glipizide-metformin,0
number_outpatient,0
acetohexamide,0
diag_2_missing,0


In [20]:
df["encounter_id"].nunique()  # Inspect the selected cleaned column

101766

## Univariate EDA after cleaning

### Number of diagnoses

In [21]:
df["number_diagnoses"].describe()  # Inspect the selected cleaned column

count    101766.000000
mean          7.422607
std           1.933600
min           1.000000
25%           6.000000
50%           8.000000
75%           9.000000
max          16.000000
Name: number_diagnoses, dtype: float64

In [22]:
show_numeric_distribution(df["number_diagnoses"], "Number of diagnoses", discrete=True)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_25564\4155684497.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Prior outpatient visits

In [23]:
df["number_outpatient"].describe()  # Inspect the selected cleaned column

count    101766.000000
mean          0.369357
std           1.267265
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          42.000000
Name: number_outpatient, dtype: float64

In [24]:
show_numeric_distribution(df["number_outpatient"], "Prior outpatient visits", discrete=True)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_25564\4155684497.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Secondary diagnosis code

In [25]:
df["diag_2"].value_counts(dropna=False).head(20).to_frame(name="count")  # Create or handle the diagnosis missingness information

,count
diag_2,
276,6752
428,6662
250,6071
427,5036
401,3736
496,3305
599,3288
403,2823
414,2650


In [26]:
show_category_counts(df["diag_2"], "Secondary diagnosis code", top_n=20)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_25564\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Readmission status

In [27]:
df["readmitted"].value_counts(dropna=False).head(10).to_frame(name="count")  # Inspect the selected cleaned column

,count
readmitted,
NO,54864
>30,35545
<30,11357


In [28]:
show_category_counts(df["readmitted"], "Readmission status", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_25564\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Admission source ID

In [29]:
df["admission_source_id"].value_counts(dropna=False).head(20).to_frame(name="count")  # Inspect the selected cleaned column

,count
admission_source_id,
7,57494
1,29565
17,6781
4,3187
6,2264
2,1104
5,855
3,187
20,161


In [30]:
show_category_counts(df["admission_source_id"], "Admission source ID", top_n=20)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_25564\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Glipizide-metformin status

In [31]:
df["glipizide-metformin"].value_counts(dropna=False).head(10).to_frame(name="count")  # Inspect the selected cleaned column

,count
glipizide-metformin,
No,101753
Steady,13


In [32]:
show_category_counts(df["glipizide-metformin"], "Glipizide-metformin status", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_25564\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Acetohexamide status

In [33]:
df["acetohexamide"].value_counts(dropna=False).head(10).to_frame(name="count")  # Inspect the selected cleaned column

,count
acetohexamide,
No,101765
Steady,1


In [34]:
show_category_counts(df["acetohexamide"], "Acetohexamide status", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_25564\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


## Numeric relationships

In [35]:
numeric_columns = ["number_diagnoses", "number_outpatient"]  # Select numeric columns for relationship analysis
df[numeric_columns].corr()  # Inspect the selected cleaned column

,number_diagnoses,number_outpatient
number_diagnoses,1.000000,0.094152
number_outpatient,0.094152,1.000000


In [36]:
correlation = df[numeric_columns].corr()  # Calculate the numeric correlation matrix
plt.figure(figsize=(5, 4))  # Create the chart area
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="Blues", vmin=-1, vmax=1)  # Plot the numeric correlation heatmap
plt.title("Correlation between utilization measures")  # Set the chart title
plt.tight_layout()  # Adjust chart spacing
plt.show()  # Display the chart

C:\Users\anind\AppData\Local\Temp\ipykernel_25564\622292652.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


## Readmission relationships

In [37]:
df.groupby("readmitted")["number_diagnoses"].agg(["count", "mean", "median", "min", "max"])  # Summarize the measure by readmission group

,count,mean,median,min,max
readmitted,,,,,
<30,11357,7.692789,9.0,1,16
>30,35545,7.646898,9.0,1,16
NO,54864,7.221366,8.0,1,16


In [38]:
plt.figure(figsize=(7, 4))  # Create the chart area
sns.boxplot(data=df, x="readmitted", y="number_diagnoses", order=["NO", ">30", "<30"], color="#72B7B2")  # Plot the feature box plot
plt.title("Number of diagnoses by readmission status")  # Set the chart title
plt.xlabel("Readmission status")  # Label the chart axis
plt.ylabel("Number of diagnoses")  # Label the chart axis
plt.tight_layout()  # Adjust chart spacing
plt.show()  # Display the chart

C:\Users\anind\AppData\Local\Temp\ipykernel_25564\2958257085.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


In [39]:
df.groupby("readmitted")["number_outpatient"].agg(["count", "mean", "median", "min", "max"])  # Summarize the measure by readmission group

,count,mean,median,min,max
readmitted,,,,,
<30,11357,0.436911,0.0,0,40
>30,35545,0.496329,0.0,0,42
NO,54864,0.273112,0.0,0,36


In [40]:
plt.figure(figsize=(7, 4))  # Create the chart area
sns.boxplot(data=df, x="readmitted", y="number_outpatient", order=["NO", ">30", "<30"], color="#F58518")  # Plot the feature box plot
plt.title("Prior outpatient visits by readmission status")  # Set the chart title
plt.xlabel("Readmission status")  # Label the chart axis
plt.ylabel("Prior outpatient visits")  # Label the chart axis
plt.tight_layout()  # Adjust chart spacing
plt.show()  # Display the chart

C:\Users\anind\AppData\Local\Temp\ipykernel_25564\3433660840.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


## Save the cleaned split

In [41]:
output_dir = Path("EDA_Dataset")  # Set the cleaned-data output folder
output_dir.mkdir(exist_ok=True)  # Set the cleaned-data output folder
output_path = output_dir / "dataset_6.csv"  # Set the cleaned CSV output path
df.to_csv(output_path, index=False)  # Save the cleaned dataframe
print(f"Saved {df.shape[0]:,} rows and {df.shape[1]:,} columns to {output_path}")  # Report the saved dataset details

Saved 101,766 rows and 9 columns to EDA_Dataset\dataset_6.csv


In [42]:
saved_df = pd.read_csv(output_path)  # Reload the saved CSV for verification
assert saved_df.shape == df.shape, "Saved CSV shape does not match the cleaned dataframe"  # Validate the cleaning result
saved_df.head()  # Reload the saved CSV for verification

,encounter_id,diag_2,readmitted,number_diagnoses,admission_source_id,glipizide-metformin,number_outpatient,acetohexamide,diag_2_missing
0,2278392,Unknown,NO,1,1,No,0,No,True
1,149190,250.01,>30,9,7,No,0,No,False
2,64410,250,NO,6,7,No,2,No,False
3,500364,250.43,NO,7,7,No,0,No,False
4,16680,157,NO,5,7,No,0,No,False
